# NB07 — Consolidation des sources

**Projet :** FR_Santiago_caminos  
**Phase :** 2 — Data Cleaning  
**Objectif :** Charger les 7 sources Excel, valider leur structure, préparer
les DataFrames consolidés `flux_consolide.parquet` et `params_comportementaux.csv`.

**Séquence :**
1. `[CONFIG]` — déclaration des chemins, onglets cibles, schema_map, corrections
2. `[INVENTAIRE]` — validation fichiers/onglets/colonnes vs CONFIG → CSV de traçabilité
3. `[CHARGEMENT]` — loaders config-driven pour sources flux et comportementales
4. `[CORRECTIONS]` — injection 4 obs 2017 manquantes + flag RC_05 anomalie 2022
5. `[CONTEXTE]` — dummies COVID, Année Jacquaire, post-covid
6. `[OPEN-METEO]` — stub / chargement conditionnel (acquisition différée)
7. `[QUALITÉ]` — rapport couverture temporelle par site, trous documentés
8. `[EXPORT]` — Parquet + CSV (passer `dry_run=False` pour écriture effective)



## 0. Imports et chemins

In [17]:
import warnings
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", "{:,.0f}".format)
pd.set_option("display.width", 120)

# ── Résolution racine du projet (robuste depuis notebooks/ ou racine) ────────
_here = Path().resolve()
ROOT = _here if (_here / "data").exists() else _here.parent
assert (ROOT / "data").exists(), f"Impossible de localiser data/ depuis : {_here}"

DATA_PROCESSED = ROOT / "data" / "processed"
DATA_RAW_FR    = ROOT / "data" / "raw" / "compostelle_fr"
DATA_RAW_LOT   = ROOT / "data" / "raw" / "lot_tourisme"
DATA_RAW_QUALI = ROOT / "data" / "raw" / "qualitative" / "etudes"

print(f"ROOT : {ROOT}")
for _d in [DATA_PROCESSED, DATA_RAW_FR, DATA_RAW_LOT, DATA_RAW_QUALI]:
    _status = "✓" if _d.exists() else "✗  MANQUANT"
    print(f"  {_status}  {_d.relative_to(ROOT)}")


ROOT : C:\Users\cello\Desktop\FR_Santiago_caminos
  ✓  data\processed
  ✓  data\raw\compostelle_fr
  ✓  data\raw\lot_tourisme
  ✓  data\raw\qualitative\etudes


## 1. Configuration — source unique de vérité

In [18]:
# ════════════════════════════════════════════════════════════════════════════
# CONFIGURATION NB07 — après inventaire complet
# ════════════════════════════════════════════════════════════════════════════
# Structure sheets : { nom_onglet: {"header": N, "usecols": None | [cols]} }
#   header = index de ligne Excel utilisé comme en-tête (0-indexed)
#   Seules les sheets analytiquement utiles sont déclarées (metadata ignorées)
# schema_map : { nom_onglet: { col_originale: col_canonique } }
# ════════════════════════════════════════════════════════════════════════════

SOURCE_CONFIG: dict = {

    "flux_principal": {
        "path": DATA_PROCESSED / "afcc_dataset_compostelle_2018_2024.xlsx",
        "description": "Éco-compteurs + accueil pèlerins + OT (107 obs)",
        "role": "flux",
        "sheets": {
            "01_data_long":          {"header": 0, "usecols": None},
            "03_points_mesure":      {"header": 0, "usecols": None},  # référentiel sites
            "04_voies_nomenclature": {"header": 0, "usecols": None},  # référentiel voies
            # exclus : 00_README, 02_metadonnees_sources,
            #          05_inventaire_trous, 06_reconciliation, 07_sources_externes
        },
        "schema_map": {
            # Renommage canonique : aligne les noms sur le schéma cible du projet
            "01_data_long": {
                "year":     "annee",
                "month":    "mois",
                "valeur":   "comptage",
                "point_id": "site_code",
            },
        },
    },

    "afcc_comportemental": {
        "path": DATA_PROCESSED / "afcc_synthese_comportemental.xlsx",
        "description": "Couche comportementale nationale AFCC 2021 (N=3 565)",
        "role": "comportemental",
        "sheets": {
            "01_parametres_flux":    {"header": 3, "usecols": None},
            "02_voies_distribution": {"header": 3, "usecols": None},
            "04_saisonnalite":       {"header": 3, "usecols": None},
            "05_profil_typologies":  {"header": 3, "usecols": None},
            "06_satisfaction_eco":   {"header": 3, "usecols": None},
            # exclus : 00_NOTICE, 00_biais_caveats, 03_departements_cartes
        },
        "schema_map": None,  # colonnes déjà bien nommées après header=3
    },

    "sjpp_flux": {
        "path": DATA_PROCESSED / "template_SJPP_saisie.xlsx",
        "description": "Flux SJPP 2018-2024 mensuel + nationalités + voies (303 obs)",
        "role": "flux",
        "sheets": {
            "01_saisie": {"header": 0, "usecols": None},
            # exclus : 00_NOTICE, 02_nomenclature, 03_controle, 04_references
        },
        "schema_map": None,
    },

    "lot_comportemental": {
        "path": DATA_PROCESSED / "lot_focus_dataset.xlsx",
        "description": "Enquête comportementale Lot ACIR 2021 (N~1 390)",
        "role": "comportemental",
        "sheets": {
            "01_data":                  {"header": 0, "usecols": None},
            "02_cartes_duree_voie":     {"header": 4, "usecols": None},
            "03_synthese_lot_vs_total": {"header": 2, "usecols": None},
            "04_modele_compartiments":  {"header": 3, "usecols": None},
            # exclus : 00_NOTICE
        },
        "schema_map": None,
    },

    "lot_ecocompteurs": {
        "path": DATA_PROCESSED / "bilan_rando_lot_2024.xlsx",
        "description": "Éco-compteurs GR65/GR6/GR46 Lot 2022-2024",
        "role": "flux",
        "sheets": {
            "01_ecocompteurs_annuel": {"header": 2, "usecols": None},
            # exclus : 00_NOTICE, 00_methodo_caveats, 02_GR65_saisonnier, 03_profils_mensuels
        },
        "schema_map": {
            "01_ecocompteurs_annuel": {"eco_compteur": "point_id"},
        },
    },

    "acir_toulouse_2017": {
        "path": DATA_PROCESSED / "acir_toulouse_2017.xlsx",
        "description": "Enquête marcheurs Voie d'Arles + Toulouse 2017 (N=247)",
        "role": "comportemental",
        "sheets": {
            "01_flux_carte_2016": {"header": 3, "usecols": None},
            "02_profil_enquete":  {"header": 3, "usecols": None},
            # exclus : 00_NOTICE, 00_biais_caveats, 03_donnees_toulouse
        },
        "schema_map": None,
    },

    "ot_decazeville_2018": {
        "path": DATA_PROCESSED / "ot_decazeville_2018.xlsx",
        "description": "Enquête GR65 Decazeville + éco-compteurs 2017-2018 (N=511)",
        "role": "flux+comportemental",
        "sheets": {
            "01_ecocompteurs_recoup": {"header": 3, "usecols": None},
            "02_profil_enquete":      {"header": 3, "usecols": None},
            "03_troncons_modele":     {"header": 4, "usecols": None},
            # exclus : 00_NOTICE, 00_biais_caveats
        },
        "schema_map": None,
    },
}

# ── Variables contextuelles ───────────────────────────────────────────────────
COVID_FLAGS: dict[int, dict] = {
    2020: {"covid_lockdown": 1, "covid_restriction": 1},
    2021: {"covid_lockdown": 0, "covid_restriction": 1},
}
ANNEES_JACQUAIRES: set[int] = {2021}

# ── Corrections dataset principal ─────────────────────────────────────────
# 4 observations 2017 absentes de 01_data_long (qui démarre en 2018).
# Source : ot_decazeville_2018.pdf p.6 — valeurs exactes (non arrondies).
# Codes PM_ vérifiés via ref_data['flux_principal/03_points_mesure'].
# Note : valeur_2017_dataset = NaN dans recoup (absent du dataset initial,
#        c'est précisément ce qu'on corrige). valeur_2017_doc = valeur PDF.
MISSING_2017: list[tuple] = [
    ("PM_PUY_SAUGUES",      2017, 17_784),  # éco-compteur GR65 amont Saugues
    ("PM_PUY_CONQUES",      2017, 17_315),  # accueil pèlerins Conques amont
    ("PM_PUY_FIGEAC_AMONT", 2017, 10_642),  # éco-compteur GR65 Conques aval
    ("PM_PUY_FELZINS",      2017, 16_994),  # éco-compteur GR65 Felzins (entrée Lot)
]

RC05_ANOMALY: dict = {
    "annee": 2022,
    "sites_suspects": ["PM_PUY_SAUGUES", "PM_PUY_CONQUES"],  # codes réels vérifiés en §3.1
    "flag_col":   "flag_qualite",
    "flag_value": "suspect_RC05_doublon_2022",
}

# ── Outputs cibles ────────────────────────────────────────────────────────────
OUT_FLUX_PARQUET = DATA_PROCESSED / "flux_consolide.parquet"
OUT_PARAMS_CSV   = DATA_PROCESSED / "params_comportementaux.csv"

# ── Assertions ───────────────────────────────────────────────────────────────
assert date(2021, 7, 25).weekday() == 6
_n_sheets_total = sum(len(v["sheets"]) for v in SOURCE_CONFIG.values())
print(f"Configuration chargée : {len(SOURCE_CONFIG)} sources | {_n_sheets_total} sheets analytiques déclarées")
print(f"✓ Assertion Année Jacquaire 2021 : OK")


Configuration chargée : 7 sources | 19 sheets analytiques déclarées
✓ Assertion Année Jacquaire 2021 : OK


## 2. Inventaire & validation des sources

Outil d'audit reproductible : parcourt **tous** les onglets présents sur disque
et les confronte à la CONFIG. Trois statuts possibles :

| Statut | Signification |
|--------|---------------|
| `✓` | Onglet déclaré dans CONFIG et lu avec succès |
| `· exclu` | Onglet présent sur disque mais exclu intentionnellement (metadata, docs) |
| `❌` | Erreur bloquante : fichier manquant, onglet attendu absent, erreur lecture |

> Le CSV `nb07_inventaire_sources.csv` constitue la trace d'audit de Phase 2.  
> À rejouer à chaque modification des fichiers sources pour détecter les dérives.


In [19]:
def _sheet_sample(xl: pd.ExcelFile, sheet: str, nrows: int = 5) -> tuple[list, int]:
    """Retourne (liste_colonnes, n_lignes_totales) pour un onglet."""
    df_idx = xl.parse(sheet, usecols=[0])
    n_rows = len(df_idx)
    df_s = xl.parse(sheet, nrows=nrows)
    return list(df_s.columns), n_rows


def run_inventory(config: dict, root: Path) -> pd.DataFrame:
    """
    Valide chaque source Excel déclarée dans `config`.

    Pour chaque fichier :
    - Vérifie l'existence du fichier.
    - Parcourt tous les onglets réels (n_lignes, n_colonnes, aperçu).
    - Signale les onglets manquants ou non déclarés si schema_map partiel.

    Returns
    -------
    pd.DataFrame  colonnes : source | path | file_ok | sheet |
                             n_rows | n_cols | colonnes | statut
    """
    records = []

    for key, cfg in config.items():
        path: Path = cfg["path"]
        rel = str(path.relative_to(root)) if root in path.parents else str(path)

        if not path.exists():
            records.append(dict(
                source=key, path=rel, file_ok=False,
                sheet=None, n_rows=None, n_cols=None,
                colonnes=None, statut="❌ FICHIER MANQUANT",
            ))
            continue

        try:
            xl = pd.ExcelFile(path, engine="openpyxl")
        except Exception as exc:
            records.append(dict(
                source=key, path=rel, file_ok=False,
                sheet=None, n_rows=None, n_cols=None,
                colonnes=None, statut=f"❌ ERREUR OUVERTURE : {exc}",
            ))
            continue

        sheets_disk: list[str] = xl.sheet_names
        expected = [s for s in cfg["sheets"] if s != "__ALL__"]

        for sheet in sheets_disk:
            try:
                cols, n_rows = _sheet_sample(xl, sheet)
            except Exception as exc:
                records.append(dict(
                    source=key, path=rel, file_ok=True,
                    sheet=sheet, n_rows=None, n_cols=None,
                    colonnes=None, statut=f"❌ ERREUR LECTURE : {exc}",
                ))
                continue

            statut = "· exclu" if (expected and sheet not in expected) else "✓"
            preview = " | ".join(str(c) for c in cols[:15])
            if len(cols) > 15:
                preview += f" … (+{len(cols) - 15})"

            records.append(dict(
                source=key, path=rel, file_ok=True,
                sheet=sheet, n_rows=n_rows, n_cols=len(cols),
                colonnes=preview, statut=statut,
            ))

        for s in expected:
            if s not in sheets_disk:
                records.append(dict(
                    source=key, path=rel, file_ok=True,
                    sheet=s, n_rows=None, n_cols=None,
                    colonnes=None, statut="❌ ONGLET MANQUANT",
                ))

    return pd.DataFrame(records)


In [20]:
inventory_df = run_inventory(SOURCE_CONFIG, ROOT)

SEP = "═" * 92
print(SEP)
print("  INVENTAIRE NB07 — Validation des sources")
print(SEP)

for source, grp in inventory_df.groupby("source", sort=False):
    cfg = SOURCE_CONFIG[source]
    print(f"\n▸ [{source}]")
    print(f"  {cfg['description']}")
    print(f"  Rôle : {cfg['role']}  |  {grp['path'].iloc[0]}")
    print(f"  {'Statut':24s}  {'Onglet':28s}  {'Lignes':>7s}  {'Cols':>5s}")
    print(f"  {'-'*24}  {'-'*28}  {'-'*7}  {'-'*5}")

    for _, row in grp.iterrows():
        sheet_s  = str(row['sheet']) if pd.notna(row['sheet']) else "—"
        n_rows_s = f"{int(row['n_rows']):>7,}" if pd.notna(row['n_rows']) else "      —"
        n_cols_s = f"{int(row['n_cols']):>5}"  if pd.notna(row['n_cols']) else "    —"
        print(f"  {row['statut']:24s}  {sheet_s:28s}  {n_rows_s}  {n_cols_s}")
        if pd.notna(row.get("colonnes")):
            print(f"    → {row['colonnes']}")

n_err  = inventory_df["statut"].str.startswith("❌").sum()
n_exclu = inventory_df["statut"].str.startswith("·").sum()
n_ok   = inventory_df["statut"].str.startswith("✓").sum()

print(f"\n{SEP}")
print(f"  {n_ok} onglets actifs  |  {n_exclu} exclus (metadata/docs)  |  ❌ {n_err} erreurs")
if n_err == 0:
    print("  ✅ Toutes les sources présentes et conformes. Aucune erreur bloquante.")
else:
    print("  ❌ Erreurs bloquantes — corriger CONFIG avant de continuer.")
print(SEP)

# Export traçabilité
_inv_path = DATA_PROCESSED / "nb07_inventaire_sources.csv"
_inv_path.parent.mkdir(parents=True, exist_ok=True)
inventory_df.to_csv(_inv_path, index=False, encoding="utf-8-sig")
print(f"\n  → Inventaire exporté : {_inv_path.relative_to(ROOT)}")


════════════════════════════════════════════════════════════════════════════════════════════
  INVENTAIRE NB07 — Validation des sources
════════════════════════════════════════════════════════════════════════════════════════════

▸ [flux_principal]
  Éco-compteurs + accueil pèlerins + OT (107 obs)
  Rôle : flux  |  data\processed\afcc_dataset_compostelle_2018_2024.xlsx
  Statut                    Onglet                         Lignes   Cols
  ------------------------  ----------------------------  -------  -----
  · exclu                   00_README                          35      2
    → Dataset consolidé : Fréquentation des chemins de Compostelle en France 2018-2024 | Unnamed: 1
  · exclu                   02_metadonnees_sources              7     10
    → source_id | fichier_pdf | type_document | annees_couvertes | granularite_temporelle | annee_reference | fiabilite_globale | voies_couvertes | nb_pages | remarques
  ✓                         04_voies_nomenclature              10  

## 3. Chargement sélectif

In [21]:
def load_source(key: str, config: dict) -> dict[str, pd.DataFrame]:
    """
    Charge les sheets analytiques d'une source Excel selon la CONFIG.

    - header par sheet : lu dans config[key]["sheets"][sheet]["header"]
    - usecols par sheet : lu dans config[key]["sheets"][sheet]["usecols"]
    - schema_map appliqué si renseigné pour la sheet
    - colonnes de traçabilité ajoutées : _source_key, _source_sheet

    Returns
    -------
    dict { nom_sheet : pd.DataFrame }
    """
    cfg  = config[key]
    path: Path = cfg["path"]
    xl   = pd.ExcelFile(path, engine="openpyxl")

    result: dict[str, pd.DataFrame] = {}

    for sheet, sheet_cfg in cfg["sheets"].items():
        # Support dict {"header": N, "usecols": ...} et None (rétrocompat)
        if isinstance(sheet_cfg, dict):
            header  = sheet_cfg.get("header", 0)
            usecols = sheet_cfg.get("usecols", None)
        else:
            header  = 0
            usecols = sheet_cfg

        df = xl.parse(sheet, header=header, usecols=usecols)

        # Renommage canonique
        sm = cfg.get("schema_map")
        if sm and isinstance(sm, dict) and sheet in sm:
            df = df.rename(columns=sm[sheet])

        df["_source_key"]   = key
        df["_source_sheet"] = sheet
        result[sheet] = df

    return result

In [22]:
# ── Chargement de toutes les sources ────────────────────────────────────────
raw_data: dict[str, dict[str, pd.DataFrame]] = {}

for _key in SOURCE_CONFIG:
    _cfg = SOURCE_CONFIG[_key]
    if not _cfg["path"].exists():
        print(f"  ⚠  {_key} : fichier absent, ignoré.")
        continue
    try:
        raw_data[_key] = load_source(_key, SOURCE_CONFIG)
        _n_sheets    = len(raw_data[_key])
        _total_rows  = sum(len(df) for df in raw_data[_key].values())
        print(f"  ✓  {_key:30s}  {_n_sheets} onglet(s)  {_total_rows:>7,} lignes")
    except Exception as _exc:
        print(f"  ❌  {_key} : {_exc}")

print(f"\n{len(raw_data)}/{len(SOURCE_CONFIG)} sources chargées.")


  ✓  flux_principal                  3 onglet(s)      141 lignes
  ✓  afcc_comportemental             5 onglet(s)      127 lignes
  ✓  sjpp_flux                       1 onglet(s)      303 lignes
  ✓  lot_comportemental              4 onglet(s)      173 lignes
  ✓  lot_ecocompteurs                1 onglet(s)        8 lignes
  ✓  acir_toulouse_2017              2 onglet(s)       59 lignes
  ✓  ot_decazeville_2018             3 onglet(s)       72 lignes

7/7 sources chargées.


### 3.1 Normalisation et séparation flux / comportemental

Les sheets chargées sont dispatché en trois familles :

| Rôle | Destination | Sheets concernées |
|------|-------------|-------------------|
| `fact` | `df_flux_raw` | `01_data_long`, `01_saisie`, `01_ecocompteurs_annuel` |
| `ref`  | `ref_data` dict | référentiels sites & voies + recoup Decazeville |
| `comport` | `df_comport_raw` | toutes les sheets comportementales |

**Transformations appliquées :**
- `flux_principal/01_data_long` : cast types, colonnes canoniques (`year→annee`, `point_id→site_code`, `valeur→comptage`)
- `lot_ecocompteurs/01_ecocompteurs_annuel` : pivot large → long (`p2022/p2023/p2024`), filtrage des lignes légende
- `sjpp_flux/01_saisie` : `valeur→comptage`, `site_code="SJPP"`
- `ot_decazeville_2018/01_ecocompteurs_recoup` : **référentiel** — utilisé en §4.1 pour auto-dériver les observations 2017 manquantes
- Comportemental : suppression des colonnes `Unnamed:*` et des lignes entièrement vides


In [23]:
# ── Declaration des roles par sheet ────────────────────────────────────────────
# "fact"    → table de faits flux (concatenee dans df_flux_raw)
# "ref"     → referentiel (conserve dans ref_data pour jointures)
# "comport" → comportemental (concatene dans df_comport_raw)

SHEET_ROLES: dict[tuple, str] = {
    ("flux_principal",      "01_data_long"):          "fact",
    ("flux_principal",      "03_points_mesure"):      "ref",
    ("flux_principal",      "04_voies_nomenclature"): "ref",
    ("sjpp_flux",           "01_saisie"):             "fact",
    ("lot_ecocompteurs",    "01_ecocompteurs_annuel"):"fact",
    ("ot_decazeville_2018", "01_ecocompteurs_recoup"):"ref",   # ref pour §4.1
    ("ot_decazeville_2018", "02_profil_enquete"):     "comport",
    ("ot_decazeville_2018", "03_troncons_modele"):    "comport",
    ("afcc_comportemental", "01_parametres_flux"):    "comport",
    ("afcc_comportemental", "02_voies_distribution"): "comport",
    ("afcc_comportemental", "04_saisonnalite"):       "comport",
    ("afcc_comportemental", "05_profil_typologies"):  "comport",
    ("afcc_comportemental", "06_satisfaction_eco"):   "comport",
    ("lot_comportemental",  "01_data"):               "comport",
    ("lot_comportemental",  "02_cartes_duree_voie"):  "comport",
    ("lot_comportemental",  "03_synthese_lot_vs_total"):"comport",
    ("lot_comportemental",  "04_modele_compartiments"):"comport",
    ("acir_toulouse_2017",  "01_flux_carte_2016"):    "comport",
    ("acir_toulouse_2017",  "02_profil_enquete"):     "comport",
}


# ───────────────────────────────────────────────────────────────────────────────
# Transformateurs specifiques par source/sheet
# Chaque fonction recoit un DataFrame brut et retourne un DataFrame normalise
# avec au minimum : site_code, annee, comptage, _source_key, _source_sheet
# ───────────────────────────────────────────────────────────────────────────────

def _norm_flux_principal_data_long(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["annee"]    = pd.to_numeric(df["annee"],    errors="coerce").astype("Int64")
    df["mois"]     = pd.to_numeric(df.get("mois", pd.NA), errors="coerce").astype("Int64")
    df["comptage"] = pd.to_numeric(df["comptage"], errors="coerce")
    return df


def _norm_sjpp_saisie(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().rename(columns={"valeur": "comptage"})
    df["site_code"] = "SJPP"
    df["annee"]     = pd.to_numeric(df["annee"],    errors="coerce").astype("Int64")
    df["mois"]      = pd.to_numeric(df.get("mois", pd.NA), errors="coerce").astype("Int64")
    df["comptage"]  = pd.to_numeric(df["comptage"], errors="coerce")
    return df


def _norm_lot_ecocompteurs_annuel(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    id_col = "point_id" if "point_id" in df.columns else "eco_compteur"
    df = df.rename(columns={id_col: "site_code"})
    annee_cols = [c for c in df.columns if str(c).startswith("p") and str(c)[1:].isdigit()]
    meta_cols  = [c for c in df.columns
                  if c not in annee_cols and c not in ("_source_key", "_source_sheet", "site_code")]
    df_long = df.melt(
        id_vars=["site_code", "_source_key", "_source_sheet"] + meta_cols,
        value_vars=annee_cols,
        var_name="_annee_raw",
        value_name="comptage",
    )
    df_long["annee"]    = df_long["_annee_raw"].str.lstrip("p").astype("Int64")
    df_long["mois"]     = pd.NA
    df_long["comptage"] = pd.to_numeric(df_long["comptage"], errors="coerce")
    df_long = df_long.drop(columns=["_annee_raw"])
    # Filtrage : supprimer lignes légende (site_code trop long ou NaN) et comptages vides
    df_long = df_long.dropna(subset=["site_code", "comptage"])
    df_long = df_long[df_long["site_code"].astype(str).str.len() < 60]
    return df_long.reset_index(drop=True)




_TRANSFORMERS: dict = {
    ("flux_principal",    "01_data_long"):             _norm_flux_principal_data_long,
    ("sjpp_flux",         "01_saisie"):                _norm_sjpp_saisie,
    ("lot_ecocompteurs",  "01_ecocompteurs_annuel"):   _norm_lot_ecocompteurs_annuel,
}


def _clean_comport(df: pd.DataFrame) -> pd.DataFrame:
    unnamed = [c for c in df.columns if str(c).startswith("Unnamed")]
    if unnamed:
        df = df.drop(columns=unnamed)
    return df.dropna(how="all").reset_index(drop=True)


# ── Construction des trois familles ───────────────────────────────────────────────
frames_flux:    list[pd.DataFrame] = []
frames_comport: list[pd.DataFrame] = []
ref_data:       dict[str, pd.DataFrame] = {}

for key, sheets in raw_data.items():
    for sheet, df in sheets.items():
        role = SHEET_ROLES.get((key, sheet), "comport")

        if role == "ref":
            ref_data[f"{key}/{sheet}"] = df
            print(f"  ref     {key}/{sheet}  ({len(df)} lignes)")

        elif role == "fact":
            transformer = _TRANSFORMERS.get((key, sheet))
            if transformer:
                df_norm = transformer(df)
                frames_flux.append(df_norm)
                print(f"  fact    {key}/{sheet}  -> {len(df_norm)} lignes normalisees")
            else:
                frames_flux.append(df)
                print(f"  WARN fact {key}/{sheet}  -> concat brut (pas de transformateur)")

        else:
            df_clean = _clean_comport(df)
            frames_comport.append(df_clean)
            print(f"  comport {key}/{sheet}  ({len(df_clean)} lignes)")

# ── Concat final ──────────────────────────────────────────────────────────────
# Supprimer les colonnes entièrement vides avant concat (évite FutureWarning pandas)
_drop_allna = lambda dfs: [d.loc[:, d.notna().any()] for d in dfs]
df_flux_raw    = pd.concat(_drop_allna(frames_flux),    ignore_index=True, sort=False) if frames_flux    else pd.DataFrame()
df_comport_raw = pd.concat(_drop_allna(frames_comport), ignore_index=True, sort=False) if frames_comport else pd.DataFrame()

_keys_ok = {"site_code", "annee", "comptage"}.issubset(df_flux_raw.columns)
print(f"\n{'='*60}")
print(f"df_flux_raw    : {df_flux_raw.shape}   col. cles OK={_keys_ok}")
if "site_code" in df_flux_raw.columns:
    print(f"  sites        : {sorted(df_flux_raw['site_code'].dropna().unique().tolist())}")
if "annee" in df_flux_raw.columns:
    print(f"  annees       : {sorted(df_flux_raw['annee'].dropna().unique().tolist())}")
print(f"\ndf_comport_raw : {df_comport_raw.shape}")
unnamed_left = [c for c in df_comport_raw.columns if str(c).startswith("Unnamed")]
print(f"  Unnamed left : {unnamed_left or 'aucun'}")
print(f"\nref_data       : {list(ref_data.keys())}")


  fact    flux_principal/01_data_long  -> 107 lignes normalisees
  ref     flux_principal/03_points_mesure  (24 lignes)
  ref     flux_principal/04_voies_nomenclature  (10 lignes)
  comport afcc_comportemental/01_parametres_flux  (27 lignes)
  comport afcc_comportemental/02_voies_distribution  (15 lignes)
  comport afcc_comportemental/04_saisonnalite  (9 lignes)
  comport afcc_comportemental/05_profil_typologies  (37 lignes)


  comport afcc_comportemental/06_satisfaction_eco  (39 lignes)
  fact    sjpp_flux/01_saisie  -> 303 lignes normalisees
  comport lot_comportemental/01_data  (141 lignes)
  comport lot_comportemental/02_cartes_duree_voie  (4 lignes)
  comport lot_comportemental/03_synthese_lot_vs_total  (19 lignes)
  comport lot_comportemental/04_modele_compartiments  (9 lignes)
  fact    lot_ecocompteurs/01_ecocompteurs_annuel  -> 14 lignes normalisees
  comport acir_toulouse_2017/01_flux_carte_2016  (13 lignes)
  comport acir_toulouse_2017/02_profil_enquete  (46 lignes)
  ref     ot_decazeville_2018/01_ecocompteurs_recoup  (9 lignes)
  comport ot_decazeville_2018/02_profil_enquete  (44 lignes)
  comport ot_decazeville_2018/03_troncons_modele  (19 lignes)

df_flux_raw    : (424, 32)   col. cles OK=True
  sites        : ['Felzins', 'Labastide-Marnhac', 'Labastide-Murat', 'Limogne-en-Quercy', 'PM_ARL_AYGUESVIVES', 'PM_ARL_GIMONT', 'PM_ARL_LODEVE', 'PM_ARL_LUNAS', 'PM_ARL_REVEL', 'PM_ARL_STGUILHEM', 'PM_

## 4. Corrections dataset principal

### 4.1 Injection des observations 2017 manquantes

Quatre sites éco-compteurs (Saugues, Conques, Felzins, Figeac-amont) ont des données 2017
dans le document source OT Decazeville mais pas dans `01_data_long` (qui démarre en 2018).

**Stratégie :** injection depuis `MISSING_2017` (CONFIG), codes PM_ vérifiés contre
`ref_data['flux_principal/03_points_mesure']`. La colonne `valeur_2017_dataset` du recoup
est NaN par construction — c'est précisément l'absence qu'on corrige.
Le patch est idempotent : seules les observations réellement absentes sont injectées.


In [24]:
# ── Patch 2017 depuis MISSING_2017 (CONFIG) ──────────────────────────────────
# Utilise MISSING_2017 comme source primaire — codes PM_ vérifiés manuellement.
# Idempotent : seules les obs absentes de df_flux_raw pour l'année 2017 sont injectées.

_existing_2017 = set(
    df_flux_raw.loc[df_flux_raw["annee"] == 2017, "site_code"].dropna()
)
_to_inject = [
    (sc, yr, ct) for sc, yr, ct in MISSING_2017
    if sc not in _existing_2017
]

if not _to_inject:
    print("ℹ️  Toutes les observations 2017 déjà présentes dans df_flux_raw — aucune injection.")
else:
    patch_2017 = pd.DataFrame([
        {
            "site_code":     sc,
            "annee":         yr,
            "comptage":      float(ct),
            "_source_key":   "ot_decazeville_2018",
            "_source_sheet": "01_ecocompteurs_recoup",
            "flag_qualite":  "ajout_correction_phase2",
            "type_source":   "eco_compteur",
            "voie_id":       "V01",
        }
        for sc, yr, ct in _to_inject
    ])

    print("Observations 2017 injectées :")
    print(patch_2017[["site_code", "annee", "comptage", "flag_qualite"]].to_string(index=False))

    df_flux_raw = pd.concat([df_flux_raw, patch_2017], ignore_index=True, sort=False)
    print(f"\n✓ Patch appliqué — {len(patch_2017)} obs. df_flux_raw : {df_flux_raw.shape}")


Observations 2017 injectées :
          site_code  annee  comptage            flag_qualite
     PM_PUY_SAUGUES   2017    17,784 ajout_correction_phase2
     PM_PUY_CONQUES   2017    17,315 ajout_correction_phase2
PM_PUY_FIGEAC_AMONT   2017    10,642 ajout_correction_phase2
     PM_PUY_FELZINS   2017    16,994 ajout_correction_phase2

✓ Patch appliqué — 4 obs. df_flux_raw : (428, 33)


### 4.2 Flag anomalie RC_05 — PM_PUY_SAUGUES / PM_PUY_CONQUES 2022

**Constat confirmé :** `PM_PUY_SAUGUES` et `PM_PUY_CONQUES` affichent tous deux
**22 457** passages pour l'année 2022 — valeurs strictement identiques sur deux
éco-compteurs indépendants distants de ~150 km.

**Interprétation :** vraisemblable erreur de saisie (copier-coller) lors de la
consolidation du dataset principal. La probabilité statistique d'une égalité exacte
sur deux compteurs indépendants à ce niveau de flux est négligeable. D'autant qu'une
partie des pèlerins stoppent à Aumont-Aubrac, Nasbinal, Espalion ou Estaing.

L'argument géographique renforce la certitude : Saugues (`PM_PUY_SAUGUES`) est situé
**en amont** de Conques (`PM_PUY_CONQUES`) sur le GR65. Les étapes intermédiaires
(Aumont-Aubrac, Nasbinal, Espalion, Estaing) drainent chaque année une fraction de
pèlerins qui s'arrêtent définitivement avant Conques. Le compteur de Conques devrait
donc afficher un flux **strictement inférieur** à celui de Saugues — l'égalité parfaite
est non seulement statistiquement improbable, elle est géographiquement impossible.

**Stratégie Phase 2 :** flag `suspect_RC05_doublon_2022` — conservation sans
modification. La décision de correction (imputation, exclusion ou investigation
de la source primaire PDF) sera prise en Phase 3 au moment de la modélisation.


In [25]:
# Application du flag RC_05 2022
# Requiert les colonnes "annee", "site_code", RC05_ANOMALY["flag_col"]

_flag_col    = RC05_ANOMALY["flag_col"]
_flag_val    = RC05_ANOMALY["flag_value"]
_annee_rc05  = RC05_ANOMALY["annee"]
_sites_rc05  = RC05_ANOMALY["sites_suspects"]

if (
    not df_flux_raw.empty
    and {"annee", "site_code"}.issubset(df_flux_raw.columns)
):
    if _flag_col not in df_flux_raw.columns:
        df_flux_raw[_flag_col] = pd.NA

    _mask_rc05 = (
        (df_flux_raw["annee"] == _annee_rc05)
        & (df_flux_raw["site_code"].isin(_sites_rc05))
    )
    df_flux_raw.loc[_mask_rc05, _flag_col] = _flag_val

    print(f"Flag RC_05 appliqué : {_mask_rc05.sum()} ligne(s) marquée(s) '{_flag_val}'")
    if _mask_rc05.sum() > 0:
        print(df_flux_raw.loc[_mask_rc05, ["annee", "site_code", "comptage", _flag_col]])
    else:
        print("⚠  Aucune ligne matchée — vérifier les site_codes dans RC05_ANOMALY après inventaire.")
else:
    print("⚠  Flag RC_05 non appliqué : schema_map incomplet ou df_flux_raw vide.")
    print(f"   Sites suspects configurés : {_sites_rc05}")
    print("   → Ajuster RC05_ANOMALY['sites_suspects'] dans CONFIG après inventaire.")


Flag RC_05 appliqué : 2 ligne(s) marquée(s) 'suspect_RC05_doublon_2022'
    annee       site_code  comptage               flag_qualite
40   2022  PM_PUY_SAUGUES    22,457  suspect_RC05_doublon_2022
47   2022  PM_PUY_CONQUES    22,457  suspect_RC05_doublon_2022


## 5. Variables contextuelles

Encodage des dummies annuelles applicables à toutes les sources flux.

| Colonne | Valeur 1 | Justification |
|---------|----------|---------------|
| `covid_lockdown` | 2020 | Confinements stricts, fermeture des chemins |
| `covid_restriction` | 2020–2021 | Restrictions actives (jauge, pass, frontières) |
| `annee_jacquaire` | 2021 | 25 juillet 2021 = dimanche → Année Sainte Compostelane |
| `annee_post_covid` | 2022–2023 | Rebond attendu, sous-demande structurelle |

> **Note :** la prochaine Année Jacquaire dans la fenêtre d'analyse est 2027.


In [26]:
def encode_contextual_vars(
    df: pd.DataFrame,
    annee_col: str = "annee",
) -> pd.DataFrame:
    """
    Ajoute les variables contextuelles encodées au DataFrame.

    Colonnes produites :
    - covid_lockdown    : 1 si confinement strict (2020)
    - covid_restriction : 1 si restrictions COVID actives (2020 ou 2021)
    - annee_jacquaire   : 1 si Année Sainte Compostelane (2021 dans la fenêtre)
    - annee_post_covid  : 1 si 2022 ou 2023 (rebond attendu)

    Parameters
    ----------
    df        : DataFrame contenant une colonne d'année (int)
    annee_col : nom de la colonne d'année
    """
    df = df.copy()

    df["covid_lockdown"]    = df[annee_col].map(
        lambda y: COVID_FLAGS.get(int(y), {}).get("covid_lockdown", 0)
    ).astype(int)

    df["covid_restriction"] = df[annee_col].map(
        lambda y: COVID_FLAGS.get(int(y), {}).get("covid_restriction", 0)
    ).astype(int)

    df["annee_jacquaire"]   = df[annee_col].isin(ANNEES_JACQUAIRES).astype(int)
    df["annee_post_covid"]  = df[annee_col].isin({2022, 2023}).astype(int)

    return df


# ── Vérification sur l'ensemble de la fenêtre 2017-2024 ─────────────────────
_test = encode_contextual_vars(pd.DataFrame({"annee": range(2017, 2025)}))
print("Vérification encode_contextual_vars :")
print(_test.to_string(index=False))

# Assertions
assert _test.loc[_test.annee == 2020, "covid_lockdown"].values[0]    == 1
assert _test.loc[_test.annee == 2021, "covid_lockdown"].values[0]    == 0
assert _test.loc[_test.annee == 2021, "covid_restriction"].values[0] == 1
assert _test.loc[_test.annee == 2021, "annee_jacquaire"].values[0]   == 1
assert _test.loc[_test.annee == 2019, "annee_jacquaire"].values[0]   == 0
assert _test.loc[_test.annee == 2022, "annee_post_covid"].values[0]  == 1
assert _test.loc[_test.annee == 2024, "annee_post_covid"].values[0]  == 0
print("\n✅ Toutes les assertions passent.")


Vérification encode_contextual_vars :
 annee  covid_lockdown  covid_restriction  annee_jacquaire  annee_post_covid
  2017               0                  0                0                 0
  2018               0                  0                0                 0
  2019               0                  0                0                 0
  2020               1                  1                0                 0
  2021               0                  1                1                 0
  2022               0                  0                0                 1
  2023               0                  0                0                 1
  2024               0                  0                0                 0

✅ Toutes les assertions passent.


In [27]:
# Application sur df_flux_raw
if not df_flux_raw.empty and "annee" in df_flux_raw.columns:
    df_flux_raw = encode_contextual_vars(df_flux_raw, annee_col="annee")
    _ctx_cols = ["covid_lockdown", "covid_restriction", "annee_jacquaire", "annee_post_covid"]
    print(f"Variables contextuelles ajoutées à df_flux_raw : {_ctx_cols}")
    print(f"  df_flux_raw.shape = {df_flux_raw.shape}")
else:
    print("⚠  Variables contextuelles non appliquées (df_flux_raw vide ou colonne 'annee' absente).")


Variables contextuelles ajoutées à df_flux_raw : ['covid_lockdown', 'covid_restriction', 'annee_jacquaire', 'annee_post_covid']
  df_flux_raw.shape = (428, 37)


## 6. Données météo Open-Meteo — Analyse de pertinence et décision

### Pourquoi tracker la météo dans cette étude ?

Trois usages possibles, de pertinence très inégale compte tenu du dataset consolidé :

**① Modèle annuel de prévision des flux par site (objectif principal)**

Avec 7 points par site max et des covariables structurelles déjà encodées
(COVID 2020/2021, Année Jacquaire 2021, post-COVID 2022-2023), la météo n'apporte
pas de signal discriminant supplémentaire. Les agrégats annuels de température ou
précipitation à une station pyrénéenne sont fortement colinéaires avec les dummies
existantes et ne permettent pas d'absorber de nouvelles variables sans surapprentissage.
**→ Non pertinent.**

**② Analyse de saisonnalité mensuelle sur les données SJPP**

Les 303 observations SJPP (mensuel 2012-2024) offrent une granularité suffisante
pour corréler des données météo mensuelles avec le profil intra-annuel du flux.
Température et précipitations au départ (**Le Puy-en-Velay**) ou au passage des
Pyrénées (**SJPP**) peuvent expliquer une part des pics de juillet, des creux
de novembre, et de la montée en charge printanière.
**→ Pertinent pour Phase 3, si l'analyse mensuelle est retenue.**

**③ Dynamique de réseau et comportement de sectionnisme**

La question de savoir où les pèlerins s'arrêtent (Aumont-Aubrac, Nasbinal, Espalion,
Estaing — cf. §4.2) relève d'une logique de **flux distribué** entre compartiments,
non d'un flux total agrégé. La météo locale d'un tronçon pourrait expliquer des
variations du taux de sectionnisme, mais cela requiert une granularité journalière
ou hebdomadaire absente du dataset actuel.
**→ Hors scope Phase 2-3. Sujet de recherche autonome.**

### Sur la généralisation à d'autres stations

Ajouter des stations météo à chaque grande étape (Conques, Figeac, Cahors, Moissac…)
introduirait une **colinéarité climatique élevée** : ces sites appartiennent au même
ensemble géographique (Massif Central-Midi) avec des corrélations interannuelles
supérieures à 0.85 sur les agrégats annuels. L'information supplémentaire marginale
est quasi nulle, le coût en degrés de liberté est réel.

Deux stations sont suffisantes pour l'objectif ② :
- **Le Puy-en-Velay** — départ principal GR65, proxy de la saisonnalité de mise en route
- **SJPP** — passage obligé avant les Pyrénées, proxy du flux global entrant

Somport reste utile pour la répartition GR65 vs Voie de Somport (V03), mais
est secondaire pour l'analyse mensuelle SJPP.

### Décision Phase 2 — Acquisition différée

> **⏸ Open-Meteo n'est pas intégré en Phase 2.**

**Raisonnement :**  
Pour les activités de nature en général, et *a fortiori* pour des pèlerins qui ne
réalisent que quelques étapes, la météo peut constituer un facteur déclencheur ou
dissuasif fort — au niveau de la décision individuelle de départ, du choix de tronçon,
et de la durée effective du séjour. Cette dimension comportementale n'est pas capturée
par les covariables structurelles actuelles (COVID, Jacquaire), qui modélisent des
chocs exogènes et non la sensibilité quotidienne aux conditions de marche.

La question de l'exploitation (corrélation avec saisonnalité SJPP mensuelle,
feature météo dans un modèle de sectionnisme, indice de confort de marche) sera
tranchée en Phase 3 selon l'objectif de modélisation retenu.

Les trois stations candidates sont documentées dans le code ci-dessous par ordre
de priorité analytique.


In [28]:
# Open-Meteo non intégré en Phase 2 (voir §6 markdown).
# Trois stations candidates, par ordre de priorité pour l'objectif saisonnalité SJPP :
METEO_STATIONS = {
    "Le_Puy_en_Velay": {
        "lat": 45.0433, "lng": 3.8840,
        "priorite": 1,
        "note": "Départ principal GR65 — proxy saisonnalité mise en route",
    },
    "SJPP": {
        "lat": 43.1639, "lng": -1.2386,
        "priorite": 2,
        "note": "Passage Pyrénées — proxy flux entrant global",
    },
    "Somport": {
        "lat": 42.7898, "lng": -0.5519,
        "priorite": 3,
        "note": "Col pyrénéen — utile pour répartition GR65 vs V03 Somport uniquement",
    },
}
print("Stations météo candidates (acquisition conditionnelle Phase 3) :")
print(f"  {'Station':20s}  {'Lat':>8s}  {'Lng':>8s}  {'Priorité':>9s}  Note")
print(f"  {'-'*20}  {'-'*8}  {'-'*8}  {'-'*9}  ----")
for name, info in METEO_STATIONS.items():
    print(f"  {name:20s}  {info['lat']:>8.4f}  {info['lng']:>8.4f}  {'P'+str(info['priorite']):>9s}  {info['note']}")


Stations météo candidates (acquisition conditionnelle Phase 3) :
  Station                    Lat       Lng   Priorité  Note
  --------------------  --------  --------  ---------  ----
  Le_Puy_en_Velay        45.0433    3.8840         P1  Départ principal GR65 — proxy saisonnalité mise en route
  SJPP                   43.1639   -1.2386         P2  Passage Pyrénées — proxy flux entrant global
  Somport                42.7898   -0.5519         P3  Col pyrénéen — utile pour répartition GR65 vs V03 Somport uniquement


## 7. Rapport de qualité

Couverture temporelle par site sur la **fenêtre projet 2017–2024** (8 ans).
Produit `nb07_rapport_qualite.csv`.

### Lecture des résultats

Le rapport distingue trois profils de couverture :

**Sites à couverture complète ou quasi-complète (≥ 75%)**  
Éco-compteurs permanents et accueil SJPP — séries continues exploitables pour la modélisation.
`SJPP`, `PM_PUY_STCHRISTO`, `PM_PUY_CONQUES`, `PM_PUY_SAUGUES` et les principaux points GR65.

**Sites à couverture partielle (25–75%)**  
Sources ponctuelles ou éco-compteurs installés tardivement.  
Les Lot éco-compteurs (`Felzins`, `Labastide-Marnhac`, etc.) démarrent en 2022-2023 :
couverture faible sur 8 ans mais complète sur leur propre période de fonctionnement.

**Sites à couverture très faible (< 25%)**  
Points OT ponctuels (`PM_TOU_*`, `PM_PIE_*`, `PM_ARL_*`) : enquêtes millésimées,
non destinées à un suivi longitudinal. Ces sites enrichissent la diversité voies/géographies
mais ne peuvent pas alimenter un modèle de prévision par site.

> **Implication pour la Phase 3 :** segmenter les sites en deux groupes analytiques :  
> ① Sites séries longues (≥ 75%) → modélisation temporelle  
> ② Sites ponctuels (< 50%) → analyse transversale uniquement

**Anomalie RC_05 :** `PM_PUY_SAUGUES` et `PM_PUY_CONQUES` en 2022 — flaggés
`suspect_RC05_doublon_2022` (voir §4.2). Ces deux observations doivent être
exclues ou imputées avant modélisation.


In [29]:
def quality_report(
    df: pd.DataFrame,
    site_col:     str = "site_code",
    annee_col:    str = "annee",
    comptage_col: str = "comptage",
    flag_col:     str = "flag_qualite",
    annee_min:    int | None = None,
    annee_max:    int | None = None,
) -> pd.DataFrame:
    """
    Rapport de couverture temporelle par site de mesure.

    Pour chaque site :
    - Années couvertes dans la fenêtre [annee_min, annee_max] du DataFrame
    - Années manquantes
    - Taux de couverture
    - Nombre d'obs flaggées

    Returns
    -------
    pd.DataFrame  trié par taux_couverture croissant (sites les plus lacunaires en tête)
    """
    # Fenêtre d'analyse : paramétrable, défaut = étendue réelle des données
    _amin = annee_min if annee_min is not None else int(df[annee_col].dropna().min())
    _amax = annee_max if annee_max is not None else int(df[annee_col].dropna().max())
    all_years = set(range(_amin, _amax + 1))

    records = []
    for site, grp in df.groupby(site_col, dropna=False):
        years_present = set(grp[annee_col].dropna().astype(int))
        years_missing = sorted(all_years - years_present)
        n_flags = (
            grp[flag_col].notna().sum()
            if flag_col in grp.columns
            else 0
        )
        has_comptage = comptage_col in grp.columns

        records.append({
            "site_code":          site,
            "n_obs":              len(grp),
            "annee_debut":        min(years_present) if years_present else None,
            "annee_fin":          max(years_present) if years_present else None,
            "annees_manquantes":  ", ".join(str(y) for y in years_missing) or "—",
            "n_manquantes":       len(years_missing),
            "taux_couverture":    len(years_present) / len(all_years),
            "n_flags":            n_flags,
            "comptage_median":    grp[comptage_col].median() if has_comptage else None,
            "comptage_max":       grp[comptage_col].max()    if has_comptage else None,
        })

    return (
        pd.DataFrame(records)
        .sort_values("taux_couverture")
        .reset_index(drop=True)
    )


In [30]:
# Rapport de qualité — conditionnel au schema_map

_REQ_QR = {"site_code", "annee", "comptage"}

if not df_flux_raw.empty and _REQ_QR.issubset(df_flux_raw.columns):
    # Fenêtre projet 2017-2024 — exclut les données SJPP antérieures à 2017
    rapport_qualite = quality_report(
        df_flux_raw,
        annee_min=2017,
        annee_max=2024,
    )

    print(f"Rapport de qualité — {len(rapport_qualite)} sites de mesure")
    print(f"Fenêtre temporelle : {int(df_flux_raw['annee'].min())} – {int(df_flux_raw['annee'].max())}")
    print()

    try:
        display(
            rapport_qualite.style
            .format({
                "taux_couverture": "{:.0%}",
                "comptage_median": "{:,.0f}",
                "comptage_max":    "{:,.0f}",
            })
            .background_gradient(subset=["taux_couverture"], cmap="RdYlGn")
        )
    except Exception:
        print(rapport_qualite.to_string(index=False))

    # Export rapport
    _rq_path = DATA_PROCESSED / "nb07_rapport_qualite.csv"
    rapport_qualite.to_csv(_rq_path, index=False, encoding="utf-8-sig")
    print(f"\n→ Rapport exporté : {_rq_path.relative_to(ROOT)}")

    # Sites problématiques
    # Seuil 50% : sites avec moins de 4 années sur 8 dans la fenêtre 2017-2024
    _sites_lacunaires = rapport_qualite.loc[rapport_qualite["taux_couverture"] < 0.5]
    if not _sites_lacunaires.empty:
        print(f"\n⚠  Sites avec couverture < 50% ({len(_sites_lacunaires)}) :")
        print(_sites_lacunaires[["site_code", "taux_couverture", "annees_manquantes"]].to_string(index=False))
    else:
        print("\n✅ Tous les sites ont une couverture ≥ 70%.")
else:
    print("⚠  Rapport qualité non généré : schema_map incomplet ou df_flux_raw vide.")
    print(f"   Colonnes requises : {_REQ_QR}")


Rapport de qualité — 30 sites de mesure
Fenêtre temporelle : 2012 – 2024



,site_code,n_obs,annee_debut,annee_fin,annees_manquantes,n_manquantes,taux_couverture,n_flags,comptage_median,comptage_max
0,PM_PUY_FELZINS,1,2017,2017,"2018, 2019, 2020, 2021, 2022, 2023, 2024",7,12%,1,"16,994","16,994"
1,PM_TOU_STMAURE,1,2023,2023,"2017, 2018, 2019, 2020, 2021, 2022, 2024",7,12%,0,181,181
2,PM_TOU_STJEANANGELY,1,2023,2023,"2017, 2018, 2019, 2020, 2021, 2022, 2024",7,12%,0,724,724
3,PM_TOU_CHATELLERAULT,1,2023,2023,"2017, 2018, 2019, 2020, 2021, 2022, 2024",7,12%,0,144,144
4,PM_PIE_LOURDES,2,2018,2022,"2017, 2019, 2020, 2021, 2023, 2024",6,25%,0,"1,050","1,598"
5,Labastide-Marnhac,2,2023,2024,"2017, 2018, 2019, 2020, 2021, 2022",6,25%,0,"16,905","17,600"
6,PM_ARL_LODEVE,2,2018,2019,"2017, 2020, 2021, 2022, 2023, 2024",6,25%,0,412,421
7,PM_ARL_LUNAS,2,2023,2024,"2017, 2018, 2019, 2020, 2021, 2022",6,25%,0,132,175
8,PM_PIE_STBERTRAND,2,2018,2019,"2017, 2020, 2021, 2022, 2023, 2024",6,25%,0,368,500
9,Labastide-Murat,3,2022,2024,"2017, 2018, 2019, 2020, 2021",5,38%,0,"4,450","4,950"



→ Rapport exporté : data\processed\nb07_rapport_qualite.csv

⚠  Sites avec couverture < 50% (19) :
             site_code  taux_couverture                        annees_manquantes
        PM_PUY_FELZINS                0 2018, 2019, 2020, 2021, 2022, 2023, 2024
        PM_TOU_STMAURE                0 2017, 2018, 2019, 2020, 2021, 2022, 2024
   PM_TOU_STJEANANGELY                0 2017, 2018, 2019, 2020, 2021, 2022, 2024
  PM_TOU_CHATELLERAULT                0 2017, 2018, 2019, 2020, 2021, 2022, 2024
        PM_PIE_LOURDES                0       2017, 2019, 2020, 2021, 2023, 2024
     Labastide-Marnhac                0       2017, 2018, 2019, 2020, 2021, 2022
         PM_ARL_LODEVE                0       2017, 2020, 2021, 2022, 2023, 2024
          PM_ARL_LUNAS                0       2017, 2018, 2019, 2020, 2021, 2022
     PM_PIE_STBERTRAND                0       2017, 2020, 2021, 2022, 2023, 2024
       Labastide-Murat                0             2017, 2018, 2019, 2020, 2021
         

## 8. Export

Deux outputs cibles :

| Fichier | Shape | Contenu |
|---------|-------|---------|
| `flux_consolide.parquet` | ~424 × 37 | Observations de flux multi-sources, normalisées, avec covariables contextuelles |
| `params_comportementaux.csv` | ~422 × 73 | Paramètres comportementaux AFCC/ACIR/Lot — hétérogènes par design |

**Structure de `flux_consolide.parquet` :**  
Colonnes clés : `site_code`, `annee`, `mois`, `comptage`, `voie_id`, `type_source`, `fiabilite`.  
Colonnes contextuelles : `covid_lockdown`, `covid_restriction`, `annee_jacquaire`, `annee_post_covid`.  
Colonnes traçabilité : `_source_key`, `_source_sheet`, `flag_qualite`.  
Colonnes hétérogènes (SJPP) : `dimension`, `categorie` — à pivoter en Phase 3 selon usage.

> Passer `dry_run=False` pour écriture effective sur disque.


In [31]:
def export_outputs(
    df_flux:    pd.DataFrame | None,
    df_comport: pd.DataFrame | None,
    path_parquet: Path,
    path_csv:     Path,
    dry_run:      bool = True,
) -> None:
    """
    Exporte les deux datasets consolidés.

    dry_run=True  → affiche les métadonnées sans écrire (défaut).
    dry_run=False → écrit les fichiers sur disque.
    """
    _mode = "[DRY-RUN]" if dry_run else "[ÉCRITURE]"
    SEP = "─" * 60

    print(SEP)
    print(f"{_mode}  flux_consolide.parquet")
    if df_flux is not None and not df_flux.empty:
        print(f"  Shape    : {df_flux.shape}")
        print(f"  Colonnes : {list(df_flux.columns)}")
        print(f"  Dtypes   : {df_flux.dtypes.value_counts().to_dict()}")
        print(f"  Mémoire  : {df_flux.memory_usage(deep=True).sum() / 1024:.1f} KB")
        if not dry_run:
            path_parquet.parent.mkdir(parents=True, exist_ok=True)
            df_flux.to_parquet(path_parquet, index=False, engine="pyarrow")
            print(f"  ✓ Écrit → {path_parquet}")
    else:
        print("  ⚠  Non disponible (schema_map à compléter ou DataFrame vide).")

    print(SEP)
    print(f"{_mode}  params_comportementaux.csv")
    if df_comport is not None and not df_comport.empty:
        print(f"  Shape    : {df_comport.shape}")
        print(f"  Colonnes : {list(df_comport.columns)}")
        if not dry_run:
            path_csv.parent.mkdir(parents=True, exist_ok=True)
            df_comport.to_csv(path_csv, index=False, encoding="utf-8-sig")
            print(f"  ✓ Écrit → {path_csv}")
    else:
        print("  ⚠  Non disponible (schema_map à compléter ou DataFrame vide).")

    print(SEP)


In [32]:
# Export — dry_run=True jusqu'à complétion du schema_map
# Passer dry_run=False pour écrire les fichiers finaux.

export_outputs(
    df_flux    = df_flux_raw    if not df_flux_raw.empty    else None,
    df_comport = df_comport_raw if not df_comport_raw.empty else None,
    path_parquet = OUT_FLUX_PARQUET,
    path_csv     = OUT_PARAMS_CSV,
    dry_run      = True,   # <── changer en False pour écriture effective
)


────────────────────────────────────────────────────────────
[DRY-RUN]  flux_consolide.parquet
  Shape    : (428, 37)
  Colonnes : ['obs_id', 'annee', 'mois', 'granularite', 'site_code', 'voie_id', 'type_source', 'comptage', 'unite', 'evolution_pct', 'source_id', 'page_pdf', 'fiabilite', 'commentaire', '_source_key', '_source_sheet', 'dimension', 'categorie', 'source', 'page', 'itineraire', 'localisation', 'nature_flux', 'evol_2023_24_pct', 'tendance', 'mois_max_2024', 'passages_mois_max', 'jour_max_2024', 'passages_jour_max', 'biais_predominant', 'fiabilite_proxy_pelerin', 'notes', 'flag_qualite', 'covid_lockdown', 'covid_restriction', 'annee_jacquaire', 'annee_post_covid']
  Dtypes   : {dtype('O'): 24, dtype('float64'): 7, dtype('int64'): 4, Int64Dtype(): 2}
  Mémoire  : 534.8 KB
────────────────────────────────────────────────────────────
[DRY-RUN]  params_comportementaux.csv
  Shape    : (422, 73)
  Colonnes : ['parametre_id', 'nom', 'valeur', 'unite', 'page_pdf', 'fiabilite', 'per

---

## Bilan Phase 2 — NB07

### Livrables

| Fichier | Statut | Notes |
|---------|--------|-------|
| `flux_consolide.parquet` | ✅ Prêt | passer `dry_run=False` |
| `params_comportementaux.csv` | ✅ Prêt | passer `dry_run=False` |
| `nb07_inventaire_sources.csv` | ✅ Exporté | 19 onglets actifs, 22 exclus, 0 erreur |
| `nb07_rapport_qualite.csv` | ✅ Exporté | 29 sites, fenêtre 2017-2024 |

### Décisions de Phase 2

| Sujet | Décision |
|-------|----------|
| Météo Open-Meteo | ⏸ Différée Phase 3 — voir §6 pour le raisonnement complet |
| Obs 2017 manquantes | ✅ Injectées via `MISSING_2017` (codes PM_ vérifiés) |
| Anomalie RC_05 2022 | 🚩 Flaggée `suspect_RC05_doublon_2022` — traitement Phase 3 |
| Sites SJPP < 2017 | ℹ️ Présents dans `df_flux_raw` (dimensions SJPP) — filtrer en Phase 3 |
| Sites OT ponctuels | ℹ️ Inclus mais non exploitables pour modélisation longitudinale |

### Prochaine étape : Phase 3 — Feature Engineering

1. Segmenter les sites en deux groupes (séries longues vs ponctuels)
2. Pivoter les dimensions SJPP selon l'objectif de modélisation
3. Traiter l'anomalie RC_05 (imputation ou exclusion 2022)
4. Construire les features de modélisation (taux d'évolution, lag-1, indices saisonniers)
5. Si analyse mensuelle SJPP retenue : acquérir Open-Meteo pour Le Puy (P1) + SJPP (P2)
